In [ ]:
import random
import os
import numpy as np
import time
import torch
import torch.nn as nn 
import torch.nn.functional as F
import pandas as pd 
import argparse
import torch.optim as optim
import torch.utils.data as data
from tensorboardX import SummaryWriter

参数设置：

In [ ]:
#设置伪随机数，方便实验结果复现
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 即使没有 CUDA，也可以设置，不会产生影响
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # 在没有 CUDA 的情况下通常将其设置为 False

In [ ]:
parser = argparse.ArgumentParser() 
parser.add_argument("--seed", type=int, default=42,help="Seed")
parser.add_argument("--lr", type=float, default=0.001, help="learning rate")
parser.add_argument("--dropout",type=float,default=0.2, help="dropout rate")
parser.add_argument("--batch_size", type=int, default=64,help="batch size for training")
parser.add_argument("--epochs", type=int,default=10,  help="training epoches")
parser.add_argument("--top_k", type=int, default=10, help="compute metrics@top_k")
parser.add_argument("--factor_num", type=int,default=64, help="predictive factors numbers in the model")
parser.add_argument("--layers",nargs='+', default=[64,64], help="layers. Note that the first layer is the concatenation of user and item embeddings. So layers[0]/2 is the embedding size.")
parser.add_argument("--num_ng", type=int,default=4, help="Number of negative samples for training set")
parser.add_argument("--num_ng_test", type=int,default=100, help="Number of negative samples for test set")
parser.add_argument("--out", default=True,help="save model or not")

args=parser.parse_known_args()[0]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter()

seed_everything(args.seed)

处理数据：

In [ ]:
"""
这个自定义数据集类的主要目的是将原始数据组织成适用于 PyTorch 模型训练和测试的形式。
通过实现这些方法，可以方便地使用 PyTorch 提供的数据加载器来加载和处理数据，以供深度学习模型使用。
"""
class Rating_Datset(torch.utils.data.Dataset):
    def __init__(self, user_list, item_list, rating_list):
        super(Rating_Datset, self).__init__()
        self.user_list = user_list
        self.item_list = item_list
        self.rating_list = rating_list

    def __len__(self):
        return len(self.user_list)

    def __getitem__(self, idx):
        user = self.user_list[idx]
        item = self.item_list[idx]
        rating = self.rating_list[idx]

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(item, dtype=torch.long),
            torch.tensor(rating, dtype=torch.float)
            )

In [ ]:
class NCF_Data(object):
    """
    Construct Dataset for NCF
    """
    def __init__(self, args, ratings1, ratings2):
        self.num_ng = args.num_ng
        self.num_ng_test = args.num_ng_test
        self.batch_size = args.batch_size
        
        self.train_ratings = ratings1
        self.test_ratings = ratings2
        random.seed(args.seed)
        self.user_item_rating_indices = self.get_user_item_matrix_indices()
        self.user_indices, self.item_incides, self.rating_data = self.user_item_rating_indices
    
    def get_user_item_matrix_indices(self):
        #build user_item_matrix
        train_ratings = self.train_ratings
        users, items, ratings = [], [], []
        for row in train_ratings.itertuples():
            users.append(row.user_id)
            items.append(row.item_id)
            ratings.append(row.rating)
        return [np.array(users), np.array(items), np.array(ratings)]

    def get_train_instance(self):
        users, items, ratings = [], [], []
        train_ratings = self.train_ratings
        # 将训练评分数据集 self.train_ratings 与负样本数据 self.negatives[['user_id', 'negative_items']] 进行合并，以获得包含正样本和负样本的数据                                              
        for row in train_ratings.itertuples():
            users.append(row.user_id)
            items.append(row.item_id)
            ratings.append(row.rating)
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)#num_workers=4改成0，4会发生broken pipe

    def get_test_instance(self):
        users, items, ratings = [], [], []
        test_ratings = self.test_ratings
        for row in test_ratings.itertuples():
            users.append(row.user_id)
            items.append(row.item_id)
            ratings.append(row.rating)
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.num_ng_test+1, shuffle=False, num_workers=0)#num_workers=4改成0

In [ ]:
class DMF(nn.Module):
    def __init__(self, args, num_users, num_items,dataset):
        super(DMF, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim= args.factor_num #MF部分的潜在因子数量32->64
        self.layers = args.layers #MLP各层大小
        
        self.user_item_indices = torch.LongTensor([dataset.user_indices.astype(int), dataset.item_incides.astype(int)])
        self.rating_data = torch.FloatTensor(dataset.rating_data)
        self.user_item_matrix = torch.sparse_coo_tensor(self.user_item_indices, self.rating_data,
                                                        torch.Size((self.num_users, self.num_items))).to_dense().to(device)

        
        self.linear_user_1 = nn.Linear(in_features=self.num_items, out_features=self.latent_dim)
        self.linear_user_1.weight.detach().normal_(0, 0.01)
        self.linear_item_1 = nn.Linear(in_features=self.num_users, out_features=self.latent_dim)
        self.linear_item_1.weight.detach().normal_(0, 0.01)
        self.fc_layers = nn.ModuleList()
        
        self.user_fc_layers = nn.ModuleList()
        for idx in range(1, len(self.layers)):
            self.user_fc_layers.append(nn.Linear(in_features=self.layers[idx - 1], out_features=self.layers[idx]))

        self.item_fc_layers = nn.ModuleList()
        for idx in range(1, len(self.layers)):
            self.item_fc_layers.append(nn.Linear(in_features=self.layers[idx - 1], out_features=self.layers[idx]))


    def forward(self, user_indices, item_indices):
        
        user = self.user_item_matrix[user_indices]
        item = self.user_item_matrix[:, item_indices].t()

        user = self.linear_user_1(user)
        item = self.linear_item_1(item)

        for idx in range(len(self.layers) - 1):
            user = F.relu(user)
            user = self.user_fc_layers[idx](user)

        for idx in range(len(self.layers) - 1):
            item = F.relu(item)
            item = self.item_fc_layers[idx](item)

        vector = torch.cosine_similarity(user, item).view(-1, 1)
        vector = torch.clamp(vector, min=1e-6, max=1)

        return vector.squeeze()

评估：

In [ ]:
def hit(ng_item, pred_items):
    if ng_item in pred_items:
        return 1
    return 0

def ndcg(ng_item, pred_items):
    if ng_item in pred_items:
        index = pred_items.index(ng_item)
        return np.reciprocal(np.log2(index+2))
    return 0

def metrics(model, test_loader, top_k, device):
    HR, NDCG = [], []

    for user, item, label in test_loader:
        user = user.to(device)
        item = item.to(device)

        predictions = model(user, item)#不同用户对不同物品的喜好程度0-1
        _, indices = torch.topk(predictions, top_k)
        recommends = torch.take(
                item, indices).cpu().numpy().tolist()

        ng_item = item[0].item() # leave one-out evaluation has only one item per user
        HR.append(hit(ng_item, recommends))
        NDCG.append(ndcg(ng_item, recommends))
    return np.mean(HR), np.mean(NDCG)

In [ ]:

n=5
MODEL_PATH = r'/JupyterCode/NCF/'

for part in range(1,n+1):
    MODEL = f'./models1/model_div{n}/div_shard{part}.pth'
    traindataset = pd.read_csv(
        f'./data/div_shard{n}/sub_train{part}.csv',
        sep=",", 
        names = ['user_id', 'item_id', 'rating'], 
        engine='python')
    testdataset = pd.read_csv(
        f'./data/div_shard{n}/sub_test{part}.csv',
        sep=",", 
        names = ['user_id', 'item_id', 'rating'], 
        engine='python')
    # construct the train and test datasets
    data = NCF_Data(args, traindataset,testdataset)
    train_loader =data.get_train_instance()
    test_loader =data.get_test_instance()

    loss_function = nn.BCELoss()

    # set the num_users, items
    num_users = 6041
    num_items = traindataset['item_id'].nunique()+1

    # set model and optimizer
    model =DMF(args, num_users, num_items,data)
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    # train, evaluation
    best_hr = 0
    
    for epoch in range(1, args.epochs+1):
        model.train() # Enable dropout (if have).
        start_time = time.time()
        
        for user, item, label in train_loader:
            user = user.to(device)
            item = item.to(device)
            label = label.to(device)

            optimizer.zero_grad()#清零梯度
            prediction = model(user, item)   
            loss = loss_function(prediction, label)
            loss.backward()
            optimizer.step()
            writer.add_scalar('loss/Train_loss', loss.item(), epoch)

        model.eval()
        HR, NDCG = metrics(model, test_loader, args.top_k, device)
        
        writer.add_scalar('Perfomance/HR@10', HR, epoch)
        writer.add_scalar('Perfomance/NDCG@10', NDCG, epoch)
        
        elapsed_time = time.time() - start_time
        print("The time elapse of epoch {:03d}".format(epoch) + " is: " + 
                time.strftime("%H: %M: %S", time.gmtime(elapsed_time)))
        print("HR: {:.5f}\tNDCG: {:.5f}".format(np.mean(HR), np.mean(NDCG)))
        
        if HR > best_hr:
            best_hr, best_ndcg, best_epoch = HR, NDCG, epoch
            if args.out:
                if not os.path.exists(MODEL_PATH):#删去了config.
                    os.mkdir(MODEL_PATH)
                torch.save(model.state_dict(), 
                    '{}'.format(MODEL))
                
    writer.close()
    print("End. Best epoch {:03d}: HR = {:.5f},NDCG = {:.5f}".format(best_epoch, best_hr, best_ndcg))
